<h1><center>Recommender Systems YSDA Course!</center></h1>
<h1><center>Семинар №2</center></h1>

<center><img src="logo.jpg" width="500" /></center>

**В этом семинаре мы:**
- Познакомимся с датасетом YAMBDA
- Ссылка на оригинальный датасет: https://huggingface.co/datasets/yandex/yambda
- Посмотрим на контест курса: https://www.kaggle.com/competitions/ysda-rec-sys-2026
- Напишем бейзлайн
- Обучим более сложные модели (CatBoost)
- Напишем несколько новых метрик оценки качества ранжирования

**Баллы за пороги:**
- 5 баллов за пробитие 0.06
- 10 баллов за ...
- 15 баллов за ...
- Топ 3 - дополнительные 10 баллов
- Топ 10 - дополнительные 5 баллов

# Imports

In [1]:
%load_ext autoreload
%autoreload 2

In [21]:
import gc
from pathlib import Path
import os
import logging
from copy import deepcopy

import numpy as np
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt

from catboost import CatBoostRanker, Pool, CatBoostClassifier
import torch
from utils.train_utils import (
    train_and_test_catboost,
    check_on_dataset,
    check_on_sampled,
    generate_features_and_negatives_local,
    generate_test_features_local,
    METRICS,
    TO_DROP,
)
from utils.utils import (
    get_dataset,
    get_test_users,
    PREPROCESSED_DIR,
    MODEL_PATH,
    FINAL_MODEL_PATH,
    recall_at_k,
)


import kagglehub
from kagglehub import KaggleDatasetAdapter


logging.basicConfig(level=logging.WARN)

SECONDS_IN_DAY = 60 * 60 * 24
APPROX_TEST_USERS_ONLY = True

# 🗄 Датасет:

In [3]:
import duckdb
from utils.yambda_dataset import YambdaDataset
from utils.utils import setup_duckdb_connection

if APPROX_TEST_USERS_ONLY:
    FULL_DATA_PATH = f"{PREPROCESSED_DIR}/initial_data_test_users.parquet"
    SAMPLED_DATA_PATH = f"{PREPROCESSED_DIR}/initial_data_test_users_sample.parquet"
else:
    FULL_DATA_PATH = f"{PREPROCESSED_DIR}/initial_data.parquet"
    SAMPLED_DATA_PATH = f"{PREPROCESSED_DIR}/initial_data_sample.parquet"

# Fraction of "listen" events to keep (to avoid memory explosion)
LISTEN_SAMPLE_FRACTION = 0.03


In [4]:
data = get_dataset(
    full_data_path=FULL_DATA_PATH,
    sampled_data_path=SAMPLED_DATA_PATH,
    approx_test_users_only=APPROX_TEST_USERS_ONLY,
    listen_sample_fraction=LISTEN_SAMPLE_FRACTION,
    deduplicate=True,
    invalidate_cache=False,
)

dataset = YambdaDataset(
    "flat",
    "5b",
    use_local=True,
    local_dir="./generated/yambda_data",
    streaming=False,
)

artists = dataset.artist_item_mapping().to_polars()
albums = dataset.album_item_mapping().to_polars()

items_metadata = artists.join(albums, on="item_id", how="left")

read parquet


In [5]:
file_path = "test_users.csv"

test_users = kagglehub.dataset_load(
    KaggleDatasetAdapter.POLARS,
    "thekabeton/ysda-recsys-2026-yambda-dataset/versions/3",
    file_path,
).collect()

In [6]:
file_path = "artist_item_mapping_small.parquet"

artists = kagglehub.dataset_load(
    KaggleDatasetAdapter.POLARS,
    "thekabeton/ysda-recsys-2026-yambda-dataset/versions/3",
    file_path,
).collect()

# 📈 Бейзлайн: самые лайкаемые треки

In [7]:
popular_tracks = (
    data.group_by("item_id")
    .agg(pl.len())
    .sort("len", descending=True)[:100]["item_id"]
    .to_list()
)

In [8]:
popular_tracks_strs = []

for i in popular_tracks:
    popular_tracks_strs.append(str(i))

ans = " ".join(popular_tracks_strs)

test_users = test_users.with_columns(pl.lit(ans).alias("item_ids"))

test_users.write_csv("baseline.csv")

# test_users # скор ~ 0.024

# 🦾 CatBoost

<center><img src="Timesplit1.svg" width="1100" /></center>


Давайте соберём какие-то фичи из данных и обучим на них градиентный бустинг. Нужно не забывать про временные лики. Нельзя давать модели видеть данные из будущего, поэтому фичи для каждого семпла должны быть посчитаны на данных из прошлого. В простейшей схеме предлагается разделить размеченые данные на 3 части:
- Вторая часть - train
- Третья часть - validation
- Первую часть используем для расчёта статистик для трейна
- Для валидации считаем статистики используя первую и вторую части вместе

#### Делим data на 3 части:

In [7]:
SPLIT_1 = 100
SPLIT_2 = 200
SPLIT_3 = 203

In [ ]:
def make_parts(
    data, day_split_1: int = 100, day_split_2: int = 200, day_split_3: int = 300
):
    data_part1 = data.filter(pl.col("timestamp") < SECONDS_IN_DAY * day_split_1)
    data_part2 = data.filter(
        (pl.col("timestamp") >= SECONDS_IN_DAY * day_split_1)
        & (pl.col("timestamp") < SECONDS_IN_DAY * day_split_2)
    )
    data_part3 = data.filter(
        (pl.col("timestamp") >= SECONDS_IN_DAY * day_split_2)
        & (pl.col("timestamp") < SECONDS_IN_DAY * day_split_3)
    )
    return data_part1, data_part2, data_part3

In [9]:
data_len_div3 = int(len(data) / 3)

target_for_fake = 0.0
data = data.sort("timestamp").with_columns(
    pl.when(pl.col("event_type").eq("like"))
    .then(1)
    .when(pl.col("event_type").eq("dislike"))
    .then(0)
    .otherwise(target_for_fake)
    .alias("target")
)


#### Набираем негативы:

In [10]:
import polars as pl
import numpy as np


def add_popular_random_negatives(
    pool_df: pl.DataFrame,
    df: pl.DataFrame,
    k: int,
    top_n: int = 100_000,
    seed: int = 42,
) -> pl.DataFrame:
    n_requested = df.height * k
    rng = np.random.default_rng(seed)

    # Get bounds and target schema
    min_ts = df.get_column("timestamp").min()
    max_ts = df.get_column("timestamp").max()
    organic_mean = pool_df.get_column("is_organic").mean()

    # Store schema for precise casting
    target_schema = df.schema

    # 1. Prepare the item pool (handling NULLs)
    top_items_pool = (
        pool_df.group_by(["item_id", "artist_id", "album_id"])
        .len()
        .sort("len", descending=True)
        .head(top_n)
        .drop("len")
    )

    # 2. Sample rows
    indices = rng.integers(0, top_items_pool.height, size=n_requested)
    sampled_metadata = top_items_pool[indices]

    # 3. Build the negative DataFrame with strict casting
    neg = pl.DataFrame(
        {
            "uid": pl.concat([df.get_column("uid")] * k),
            "timestamp": pl.Series(
                rng.integers(min_ts, max_ts, size=n_requested),
                dtype=target_schema["timestamp"],
            ),
            "is_organic": pl.Series(rng.random(size=n_requested) < organic_mean).cast(
                target_schema["is_organic"]
            ),  # Cast to match (e.g., Boolean vs UInt8)
            "event_type": pl.repeat("random_negative", n_requested, eager=True),
            "target": pl.repeat(0, n_requested, eager=True).cast(
                target_schema["target"]
            ),
        }
    )

    # 4. Attach sampled item metadata
    # We must ensure item_id, artist_id, and album_id also match the schema
    sampled_metadata = sampled_metadata.with_columns(
        [
            pl.col("item_id").cast(target_schema["item_id"]),
            pl.col("artist_id").cast(target_schema["artist_id"]),
            pl.col("album_id").cast(target_schema["album_id"]),
        ]
    )

    neg = pl.concat([neg, sampled_metadata], how="horizontal")

    # 5. Filter and Concatenate
    cols = df.columns
    neg_filtered = neg.join(
        df.select(["uid", "item_id"]), on=["uid", "item_id"], how="anti"
    ).select(cols)

    return pl.concat([df.select(cols), neg_filtered], how="vertical")

#### Проделываем то-же самое для валидации. Фичи считаем по событиям из 2 части датасета. Затем клеим их к 3 части:

In [11]:
from utils.data_preprocessing_pipeline import DataPreprocessingPipeline

fstr_dict = {
    "uid_listens_90d": 71.6720133906344,
    "item_all_life": 14.999871445868624,
    "item_all_30d": 8.318400888547968,
    "item_likes_life": 3.203577265056065,
    "uid_listens_life": 0.8133637687467504,
    "uid_like_all_ratio_90d": 0.7277936238767376,
    "item_cnt_30d": 0.11285190197427988,
    "item_org_likes_7d": 0.10117153065597632,
    "uid_artist_like_ratio_30d": 0.05095618463919989,
    "artist_id": 0.0,
    "album_id": 0.0,
    "item_listens_life": 0.0,
    "item_org_all_life": 0.0,
}
feature_columns = list(fstr_dict.keys())
# fstr_dict = None

pipeline = DataPreprocessingPipeline(
    random_negatives_configs=[],
    feature_windows=["7d", "30d", "90d"],
    feature_importances=fstr_dict,
    feature_importance_threshold=0.01,
    target_for_fake=0.00,
    preprocess_dir=".tmp",
    preprocess_prefix="test_pipeline_consistency",
    use_cache=False,
    invalidate_cache=True,
    seed=42,
)


data_part1, data_part2, data_part3 = make_parts(
    data, day_split_1=SPLIT_1, day_split_2=SPLIT_2, day_split_3=SPLIT_3
)
data_part2 = add_popular_random_negatives(data_part1, data_part2, k=10, top_n=100_000)

pipeline_df = pipeline.generate_train_features(
    pl.concat([data_part1, data_part2, data_part3], how="vertical")
)

[Subprocess] Starting feature generation
[Subprocess] History path: .tmp/test_pipeline_consistency_train_with_negatives.parquet
[Subprocess] To enrich path: None
[Subprocess] Created FeatureGenerator
Processing features for prefix: item...
Processing features for prefix: uid...
Processing features for prefix: artist...
Processing features for prefix: album...
Processing features for prefix: uid_artist...
Processing features for prefix: uid_item...
Processing features for prefix: uid_album...
[Subprocess] Generated features, shape (1457742, 18)
[Subprocess] Saved result


In [12]:
pipeline_data_len_div_3 = int(len(pipeline_df) / 3)

# assert pipeline_data_len_div_3 == data_len_div3

pipeline_part1, pipeline_part2, pipeline_part3 = make_parts(
    pipeline_df, day_split_1=SPLIT_1, day_split_2=SPLIT_2, day_split_3=SPLIT_3
)

#### Обучаем катбуст:

In [13]:
train = pipeline_part2
val = pipeline_part3
# val = val.filter(pl.col("event_type").is_in(["like", "dislike"]))

In [14]:
to_drop = [column for column in TO_DROP if column in train.columns]

MAX_GROUP_SIZE = 1023
train = train.group_by("uid").head(MAX_GROUP_SIZE)
train_pool = Pool(
    data=train.drop(to_drop),
    label=train["target"],
    group_id=train["uid"],
)

val = val.group_by("uid").head(MAX_GROUP_SIZE)
val_pool = Pool(
    data=val.drop(to_drop),
    label=val["target"],
    group_id=val["uid"],
)

model = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.1,
    depth=4,
    l2_leaf_reg=10,
    loss_function="CrossEntropy",
    eval_metric="CrossEntropy",
    custom_metric=METRICS,
    early_stopping_rounds=200,
    verbose=10,
    use_best_model=True,
    task_type="GPU",
)

model = model.fit(train_pool, eval_set=val_pool, plot=True)

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

Default metric period is 5 because AUC, LogLikelihoodOfPrediction, QueryAUC, RecallAt is/are not implemented for GPU
Metric QueryAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric RecallAt is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric LogLikelihoodOfPrediction is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	learn: 0.4794659	test: 0.5969931	best: 0.5969931 (0)	total: 19.3ms	remaining: 38.6s
10:	learn: 0.0965746	test: 0.6067981	best: 0.5167616 (3)	total: 76.8ms	remaining: 13.9s
20:	learn: 0.0811899	test: 0.6851545	best: 0.5167616 (3)	total: 136ms	remaining: 12.9s
30:	learn: 0.0790080	test: 0.7050314	best: 0.5167616 (3)	total: 194ms	remaining: 12.3s
40:	learn: 0.0780736	test: 0.7088572	best: 0.5167616 (3)	total: 252ms	remaining: 12s
50:	learn: 0.0774036	test: 0.7075534	best: 0.5167616 (3)	total: 308ms	remaining: 11.8s
60:	learn: 0.0768892	test: 0.7032909	best: 0.5167616 (3)	total: 364ms	remaining: 11.6s
70:	learn: 0.0766827	test: 0.7026276	best: 0.5167616 (3)	total: 421ms	remaining: 11.4s
80:	learn: 0.0764425	test: 0.7002464	best: 0.5167616 (3)	total: 477ms	remaining: 11.3s
90:	learn: 0.0762822	test: 0.6994615	best: 0.5167616 (3)	total: 533ms	remaining: 11.2s
100:	learn: 0.0761735	test: 0.6987466	best: 0.5167616 (3)	total: 590ms	remaining: 11.1s
110:	learn: 0.0760423	test: 0.6974937	best:

#### Важности фичей:

In [15]:
imps = model.get_feature_importance(type="PredictionValuesChange")
pairs = sorted(zip(model.feature_names_, imps), key=lambda x: x[1], reverse=True)

for name, val in pairs:
    print(f"{name}: {val}")

uid_listens_90d: 40.819593065413464
uid_listens_life: 25.95168381459073
item_all_life: 23.71521715046062
item_all_30d: 3.6698799973941116
item_likes_life: 2.5521024025885817
item_cnt_30d: 2.3874428657297972
uid_like_all_ratio_90d: 0.9040807038227068
artist_id: 0.0
album_id: 0.0
item_org_likes_7d: 0.0
uid_artist_like_ratio_30d: 0.0


### 🔍  Retrieval:

#### Кандидатогенератор популярных треков

In [16]:
def enrich_test_df(
    test_df: pl.DataFrame,
    data_part2: pl.DataFrame,
    data_part3: pl.DataFrame,
    seed: int = 42,
) -> pl.DataFrame:
    """
    Enriches test_df with metadata from data_part2 and temporal range from data_part3.
    """
    rng = np.random.default_rng(seed)
    n_rows = test_df.height
    master_schema = data_part2.schema

    item_metadata = data_part2.select(["item_id", "artist_id", "album_id"]).unique(
        subset=["item_id"]
    )

    min_ts = data_part3.get_column("timestamp").min()
    max_ts = data_part3.get_column("timestamp").max()

    synthetic_cols = pl.DataFrame(
        {
            "timestamp": pl.Series(
                rng.integers(min_ts, max_ts, size=n_rows),
                dtype=data_part3.schema["timestamp"],
            ),
            "is_organic": pl.repeat(False, n_rows, eager=True).cast(
                data_part2.schema["is_organic"]
            ),
            "event_type": pl.repeat("candidate", n_rows, eager=True),
            "target": pl.repeat(0, n_rows, eager=True).cast(
                data_part2.schema["target"]
            ),
        }
    )

    enriched = test_df.join(item_metadata, on="item_id", how="left")

    enriched = pl.concat([enriched, synthetic_cols], how="horizontal")

    enriched = enriched.select(
        [pl.col(col).cast(dtype) for col, dtype in master_schema.items()]
    )

    return enriched

In [17]:
popular_tracks = (
    data.group_by("item_id")
    .agg(pl.len())
    .sort("len", descending=True)[:1000]
    .select(["item_id"])
)

In [18]:
test = test_users.select(["uid"]).join(popular_tracks, how="cross")

In [19]:
test = enrich_test_df(test, data_part2, data_part3)

#### Считаем фичи

In [22]:
positive_interactions = data_part3.filter(pl.col("event_type").eq("like")).select(
    ["uid", "item_id"]
)

In [20]:
pipeline_df = pipeline.generate_train_features(
    pl.concat([data_part1, data_part2, test], how="vertical")
)

_, _, test_with_features = make_parts(
    pipeline_df, day_split_1=SPLIT_1, day_split_2=SPLIT_2
)

[Subprocess] Starting feature generation
[Subprocess] History path: .tmp/test_pipeline_consistency_train_with_negatives.parquet
[Subprocess] To enrich path: None
[Subprocess] Created FeatureGenerator
Processing features for prefix: item...
Processing features for prefix: uid...
Processing features for prefix: artist...
Processing features for prefix: album...
Processing features for prefix: uid_artist...
Processing features for prefix: uid_item...
Processing features for prefix: uid_album...
[Subprocess] Generated features, shape (27474335, 18)
[Subprocess] Saved result


#### Применяем модель

In [ ]:
test_pool = Pool(
    data=test_with_features.drop(
        [column for column in to_drop if column in test_with_features.columns]
    ),
)

In [32]:
scores = model.predict_proba(test_pool)[:, 1]
scores_pl = (
    test_with_features.select(["uid", "item_id"])
    .with_columns(pl.Series("score", scores))
    .sort(["uid", "score"], descending=[False, True])
    .with_columns(
        [
            pl.col("score")
            .rank(method="ordinal", descending=True)
            .over("uid")
            .alias("rank")
        ]
    )
    .sort(["uid", "rank"])
)

for k in [100, 300, 1000]:
    recall_k = recall_at_k(
        positive_interactions=positive_interactions,
        candidates=scores_pl,
        k=k,
    )
    print(f"Recall@{k}: {recall_k}")

Recall@100: 0.009072645143127694
Recall@300: 0.022460875017595045
Recall@1000: 0.0523756510166737


In [ ]:
pred = model.predict_proba(test_pool)[:, 1]

submit = (
    test.select(["uid", "item_id"])
    .with_columns(pl.Series("pred", pred))
    .sort(["uid", "pred"], descending=[False, True])
    .group_by("uid")
    .agg(pl.col("item_id").head(100).cast(pl.Utf8).str.join(" ").alias("item_ids"))
)

submit

uid,item_ids
i64,str
89,"""7599492 3971215 4899017 736814…"
153,"""7599492 3971215 4899017 647068…"
164,"""7599492 3971215 6470688 736814…"
216,"""3971215 7599492 7368148 489901…"
291,"""3971215 7599492 6470688 489901…"
…,…
999735,"""7599492 3971215 7368148 647068…"
999737,"""3971215 7599492 7368148 489901…"
999779,"""7599492 3971215 6470688 489901…"


In [ ]:
submit.write_csv("catboost.csv")  # скор ~ 0.043

### Что дальше?

- Правильная оффлайн валидация (За какие даты собран тест?)
- Правильно собранный пул для обучения
- Больше фичей (Как сделать фичи из эмбеддингов?)
- Более богатые негативы
- Более богатые кандидатогенераторы
- CatBoostClassifier?
- Гиперпараметры модели
- Больше данных
- Учиться на всех данных
<center><img src="Timesplit2.svg" width="1100" /></center>